# Day 2 – Module 3: Embeddings, Semantic Similarity and Vector Databases

**Topics covered:**
- Concept of embeddings
- Semantic similarity and cosine distance
- Use cases in search and retrieval
- Purpose of vector databases
- FAISS and Chroma overview
- Retrieval pipelines for semantic search

**What you will do in this notebook:**
Generate text embeddings with sentence-transformers, compute semantic similarity, build a FAISS index and a Chroma collection, and run end-to-end retrieval pipelines. Every code cell runs locally — no API keys required. The embedding model name comes from `config.json`.

## 1. How embeddings work

Embeddings convert text into fixed-size numeric vectors (lists of floating-point numbers). Texts with similar meaning produce vectors that are close together in vector space.

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        EMBEDDING FLOW                                   │
│                                                                         │
│  "How do I reset my password?"                                          │
│       │                                                                 │
│       ▼                                                                 │
│  ┌─────────────────────┐                                                │
│  │  Embedding Model     │   sentence-transformers / HF                  │
│  │  (all-MiniLM-L6-v2) │   384 dimensions per vector                   │
│  └──────────┬──────────┘                                                │
│             ▼                                                           │
│  [0.032, -0.118, 0.045, ..., 0.091]   (384 floats)                     │
│                                                                         │
│  Similar meaning → similar vectors → small cosine distance              │
│  Different meaning → different vectors → large cosine distance          │
└─────────────────────────────────────────────────────────────────────────┘
```

| Property | Value |
|----------|-------|
| Input | Any text (sentence, paragraph, document chunk) |
| Output | Fixed-size vector (e.g. 384 or 768 dimensions) |
| Model | sentence-transformers (`all-MiniLM-L6-v2` — fast, 384d) |
| Similarity metric | Cosine similarity or L2 (Euclidean) distance |

## 2. Setup

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

with open(Path.cwd() / "config.json") as f:
    CONFIG = json.load(f)

print("config.json loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

print(f"numpy:  {np.__version__}")
print(f"torch:  {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Load the embedding model from config
embed_model = SentenceTransformer(CONFIG["embedding_model"])

print(f"Model:      {CONFIG['embedding_model']}")
print(f"Max tokens: {embed_model.max_seq_length}")
print(f"Dimensions: {embed_model.get_sentence_embedding_dimension()}")

## 3. Generating embeddings

The `model.encode()` method converts text to vectors. Pass a single string or a list of strings.

In [ ]:
# Embed a single sentence
text = "How do I reset my password?"
vector = embed_model.encode(text)

print(f"Text:   {text}")
print(f"Vector: shape={vector.shape}, dtype={vector.dtype}")
print(f"First 10 values: {vector[:10].round(4).tolist()}")

In [ ]:
# Embed multiple sentences at once
texts = [
    "How do I reset my password?",
    "I forgot my login credentials.",
    "What are your business hours?",
    "The quarterly revenue exceeded expectations.",
]

vectors = embed_model.encode(texts)
print(f"Input:  {len(texts)} texts")
print(f"Output: shape={vectors.shape}  ({vectors.shape[0]} vectors, {vectors.shape[1]} dimensions)")

## 4. Semantic similarity

In [ ]:
from numpy.linalg import norm

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (norm(a) * norm(b)))


# Compute pairwise similarity
print("Pairwise cosine similarity:\n")
for i in range(len(texts)):
    for j in range(i + 1, len(texts)):
        sim = cosine_similarity(vectors[i], vectors[j])
        print(f"  {sim:.4f}  '{texts[i][:40]}' vs '{texts[j][:40]}'")

In [ ]:
# Query: find the most similar text to a query
query = "I can't log in to my account"
query_vec = embed_model.encode(query)

similarities = [cosine_similarity(query_vec, v) for v in vectors]

print(f"Query: '{query}'\n")
ranked = sorted(zip(similarities, texts), reverse=True)
for score, text in ranked:
    print(f"  {score:.4f}  {text}")

In [ ]:
# Similarity matrix — all pairs at once using matrix multiplication
from sentence_transformers import util

sim_matrix = util.cos_sim(vectors, vectors)

print("Similarity matrix (4x4):\n")
# Header
header = "".join(f"  [{i}]  " for i in range(len(texts)))
print(f"      {header}")
for i, row in enumerate(sim_matrix):
    vals = "".join(f" {v:.3f} " for v in row)
    print(f"  [{i}] {vals}  {texts[i][:35]}")

## 5. Use cases for embeddings in enterprise

| Use case | How embeddings help | Example |
|----------|-------------------|---------|
| **Semantic search** | Query by meaning, not keywords | "password help" matches "credential reset guide" |
| **Duplicate detection** | Similar vectors = potential duplicates | De-duplicate support tickets before routing |
| **Clustering** | Group similar documents | Cluster customer feedback into themes |
| **Recommendation** | Find items similar to user history | Recommend articles based on reading history |
| **Classification** | Nearest-neighbor classification | Classify a new ticket by similarity to labeled examples |

In [ ]:
# Use case demo: duplicate detection
tickets_dedup = [
    "Cannot access the VPN from home network",
    "VPN connection fails when working remotely",
    "How to configure email on mobile device",
    "Unable to connect to VPN from home",
    "Need to set up email on my phone",
]

vecs_dedup = embed_model.encode(tickets_dedup)
sim_dedup = util.cos_sim(vecs_dedup, vecs_dedup)

THRESHOLD = 0.80
print(f"Potential duplicates (cosine > {THRESHOLD}):\n")
for i in range(len(tickets_dedup)):
    for j in range(i + 1, len(tickets_dedup)):
        if sim_dedup[i][j] > THRESHOLD:
            print(f"  {sim_dedup[i][j]:.3f}  '{tickets_dedup[i]}' ~ '{tickets_dedup[j]}'")

In [ ]:
# Use case demo: nearest-neighbor classification
# Given labeled examples, classify a new ticket by nearest neighbor
labeled = [
    ("Cannot print from the shared printer", "hardware"),
    ("Laptop screen is flickering", "hardware"),
    ("Forgot my password", "account"),
    ("Need to reset MFA token", "account"),
    ("Application crashes on startup", "software"),
    ("Excel freezes when opening large files", "software"),
]

labeled_texts = [t for t, _ in labeled]
labeled_labels = [l for _, l in labeled]
labeled_vecs = embed_model.encode(labeled_texts)

new_ticket = "My monitor keeps going black"
new_vec = embed_model.encode(new_ticket)

sims = util.cos_sim(new_vec, labeled_vecs)[0]
best_idx = int(sims.argmax())

print(f"New ticket: '{new_ticket}'")
print(f"Nearest:    '{labeled_texts[best_idx]}' (sim={sims[best_idx]:.3f})")
print(f"Predicted:  {labeled_labels[best_idx]}")

## 6. Vector databases — why and when

Computing similarity against all vectors works for small datasets (hundreds). For larger datasets (millions of documents), you need:

- **Fast approximate nearest neighbor (ANN)** search
- **Persistence** — don't re-embed on every restart
- **Metadata filtering** — filter by category, date, etc. during search

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                    RETRIEVAL PIPELINE                                       │
│                                                                             │
│  Block 1: EMBED QUERY            Block 2: SEARCH INDEX         Block 3:    │
│                                                                  RETURN    │
│  query = "reset password"        vector DB searches for         Top-K      │
│       │                          nearest neighbors              documents  │
│       ▼                               │                              │     │
│  query_vec = model.encode(q)     ┌────▼─────┐                       ▼     │
│  [0.03, -0.12, ...]             │  Vector   │    IDs + scores    Fetch &  │
│       │                         │  Index    │───────────────►   display   │
│       └────────────────────────►│  (FAISS / │                            │
│                                  │  Chroma)  │                            │
│                                  └──────────┘                             │
│                                                                             │
│  This notebook implements all three blocks for both FAISS and Chroma.       │
└─────────────────────────────────────────────────────────────────────────────┘
```

| Feature | In-memory (numpy) | FAISS | Chroma |
|---------|-------------------|-------|--------|
| Scale | ~1K docs | Millions | Millions |
| Speed | O(n) brute force | ANN with indexing | ANN with indexing |
| Persistence | No | Manual save/load | Built-in |
| Metadata filter | Manual | Manual | Built-in |
| Embedding | External | External | Built-in or external |

## 7. FAISS — Facebook AI Similarity Search

In [ ]:
import faiss

# Sample documents — enterprise knowledge base articles
documents = [
    "To reset your password, go to Settings > Security > Change Password. You will receive a verification email.",
    "VPN access requires installing the company VPN client and authenticating with your corporate credentials and MFA token.",
    "To request new hardware, submit a ticket through the IT portal under Hardware Requests. Approvals take 3-5 business days.",
    "The company email system uses Microsoft 365. Configure mobile email using the Outlook app with your corporate credentials.",
    "Expense reports must be submitted within 30 days of travel using the expense management portal. Attach all receipts.",
    "To access the internal wiki, navigate to wiki.company.internal from any device connected to the corporate network or VPN.",
    "Software installation requests go through the IT portal. Standard software is pre-approved; non-standard requires manager sign-off.",
    "Two-factor authentication (MFA) is required for all corporate accounts. Use the authenticator app to generate time-based codes.",
    "The shared drive is available at \\\\fileserver\\shared. Mount it using Map Network Drive in File Explorer.",
    "For printer issues, check the printer queue and restart the print spooler service. Contact IT if the issue persists.",
    "Conference room booking is done through the calendar system. Rooms with video conferencing equipment are marked with a camera icon.",
    "The corporate guest WiFi network is 'Company-Guest'. It provides internet access only; no access to internal resources.",
]

print(f"Documents: {len(documents)}")

In [ ]:
# Block 1: Embed all documents
doc_embeddings = embed_model.encode(documents, convert_to_numpy=True)

# Ensure float32 (FAISS requirement)
doc_embeddings = doc_embeddings.astype("float32")

print(f"Embeddings shape: {doc_embeddings.shape}")
print(f"Dtype: {doc_embeddings.dtype}")

In [ ]:
# Block 2: Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print(f"FAISS index type: IndexFlatL2")
print(f"Dimension:        {dimension}")
print(f"Vectors stored:   {index.ntotal}")

In [ ]:
# Block 2+3: Search — embed query, search index, return results
query = "How do I change my password?"
query_vec = embed_model.encode([query]).astype("float32")

k = 3
distances, indices = index.search(query_vec, k)

print(f"Query: '{query}'")
print(f"Top-{k} results:\n")
for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"  {rank+1}. [dist={dist:.4f}]  {documents[idx]}")

In [ ]:
# Multiple queries at once
queries = [
    "I need to connect to the office VPN",
    "Where do I submit expense reports?",
    "How to book a meeting room?",
]

query_vecs = embed_model.encode(queries).astype("float32")
D, I = index.search(query_vecs, k=2)

for q, dists, idxs in zip(queries, D, I):
    print(f"Query: '{q}'")
    for dist, idx in zip(dists, idxs):
        print(f"  [dist={dist:.4f}]  {documents[idx][:80]}")
    print()

## 8. Chroma — persistent vector database

Chroma combines embedding, indexing, and search in one API

In [ ]:
import chromadb

# Create an in-memory client (use PersistentClient for disk persistence)
chroma_client = chromadb.Client()

print(f"Chroma client: {type(chroma_client).__name__}")

In [ ]:
# Create a collection with Chroma's default embedding function
collection = chroma_client.create_collection(
    name=CONFIG["chroma_collection_name"],
    metadata={"hnsw:space": "cosine"},  # use cosine distance
)

# Add documents with IDs and optional metadata
doc_ids = [f"doc_{i}" for i in range(len(documents))]
doc_metadata = [
    {"category": "account"}, {"category": "network"}, {"category": "hardware"},
    {"category": "email"}, {"category": "finance"}, {"category": "network"},
    {"category": "software"}, {"category": "account"}, {"category": "storage"},
    {"category": "hardware"}, {"category": "facilities"}, {"category": "network"},
]

collection.add(
    documents=documents,
    ids=doc_ids,
    metadatas=doc_metadata,
)

print(f"Collection: {collection.name}")
print(f"Documents:  {collection.count()}")

In [ ]:
# Query with text (Chroma embeds the query internally)
results = collection.query(
    query_texts=["How do I change my password?"],
    n_results=3,
)

print("Query: 'How do I change my password?'\n")
for doc, dist, meta, doc_id in zip(
    results["documents"][0],
    results["distances"][0],
    results["metadatas"][0],
    results["ids"][0],
):
    print(f"  [{doc_id}]  dist={dist:.4f}  cat={meta['category']:10s}  {doc[:70]}")

## 9. End-to-end retrieval pipeline

**Scenario:** Build a semantic search system for an internal support knowledge base. Given a user question, retrieve the top-2 most relevant articles.

This maps directly to the retrieval pipeline diagram:
1. **Block 1**: Embed all documents + embed query (sentence-transformers)
2. **Block 2**: Search the index (FAISS)
3. **Block 3**: Map indices to documents, return ranked results

In [ ]:
def semantic_search(
    query: str,
    documents: list[str],
    index: faiss.Index,
    model: SentenceTransformer,
    k: int = 3,
) -> list[dict]:
    """End-to-end semantic search: embed query, search FAISS, return ranked docs."""
    # Block 1: embed query
    q_vec = model.encode([query]).astype("float32")
    
    # Block 2: search index
    distances, indices = index.search(q_vec, k)
    
    # Block 3: map to documents
    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        results.append({
            "rank": rank + 1,
            "document": documents[idx],
            "distance": float(dist),
            "index": int(idx),
        })
    return results


# Test the pipeline
test_queries = [
    "I forgot my password and need to log in",
    "How do I get a new laptop for work?",
    "Where is the shared file server?",
    "Can visitors use the WiFi?",
]

for q in test_queries:
    results = semantic_search(q, documents, index, embed_model, k=2)
    print(f"Q: {q}")
    for r in results:
        print(f"  {r['rank']}. [dist={r['distance']:.4f}]  {r['document'][:70]}")
    print()

## 10. Frameworks quick reference

| Library | Purpose | Key functions | Example |
|---------|---------|---------------|---------|
| `sentence-transformers` | Text to vectors | `SentenceTransformer()`, `.encode()` | `model.encode(["text"])` |
| `faiss-cpu` | Fast vector search | `IndexFlatL2()`, `.add()`, `.search()` | `index.search(vec, k=3)` |
| `chromadb` | Persistent vector DB | `Client()`, `.create_collection()`, `.query()` | `collection.query(query_texts=["q"])` |
| `numpy` | Vector math | `np.dot()`, `norm()` | `np.dot(a, b) / (norm(a) * norm(b))` |